# VoiceStudio on Google Colab

Run the full [VoiceStudio](https://github.com/debpalash/VoiceStudio) app — voice cloning, voice design, video dubbing, and TTS in 646 languages — on a free Colab GPU, web UI included.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the notebook also works on CPU, but generation is much slower).

Run the cells top to bottom. Every cell is idempotent — re-running after a hiccup is always safe.

**Contents**

- **Part 1 — Setup & launch (cells 1-7):** GPU check, install, backend launch, open the web UI, API smoke test.
- **Part 2 — Feature tour (cells 8-20):** each major feature as a self-contained API demo with inline playback — multilingual TTS, voice cloning, voice design, voice profiles, transcription, AI-watermark detection, the OpenAI-compatible API, a multi-voice story, a chaptered audiobook, video dubbing, and vocal isolation.
- **Part 3 — Troubleshooting.**

**How this notebook works (and why):**

| Piece | Approach | Why |
|---|---|---|
| Backend install | `uv pip install --system .` | Mirrors the official Docker image: installs into Colab's Python and keeps Colab's preinstalled CUDA PyTorch when it matches our pinned torch, instead of re-downloading multi-GB wheels into a venv. The pins (`deploy/torch-constraints.txt`) are passed explicitly so torch, torchaudio and torchvision cannot drift apart (#1357). |
| Web UI | Built in-notebook with `bun` (~2 min, once per session) | Official releases ship desktop installers only — there is no prebuilt standalone web bundle to download. The backend serves the built SPA itself from `frontend/dist`. |
| Opening the app | Colab's built-in kernel port proxy | No third-party tunnel binaries; the URL is private to your Google session. (A `cloudflared` public-URL alternative is documented in the launch cell.) |

Issues with this notebook are VoiceStudio issues — report them at [debpalash/VoiceStudio/issues](https://github.com/debpalash/VoiceStudio/issues).


## Part 1 — Setup & launch


In [3]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


Wed Oct  7 17:28:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# ── 2. Clone + install (idempotent; first run ~5-8 min) ─────────────────────
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/debpalash/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts: Colab -> PyPI is usually fast, but model/wheel
# CDNs occasionally stall and uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    env = os.environ.copy()
    # Ensure bun is in the PATH for subprocesses
    bun_dir = os.path.dirname(BUN)
    if bun_dir not in env["PATH"]:
        env["PATH"] = bun_dir + os.pathsep + env["PATH"]

    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), env=env).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/debpalash/VoiceStudio/issues"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} — reusing it.")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. bun — builds the web UI (see the intro cell for why we build it here)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# 2d. Build the frontend (skipped once frontend/dist/index.html exists)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist already built — skipping.")
else:
    # The repo is a bun workspace: install at the repo root, build in web mode.
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "build:web"], cwd=REPO_DIR, what="frontend build (web)")

    if not os.path.exists(dist_index):
        raise SystemExit(
            "The frontend build finished but frontend/dist/index.html is missing —\n"
            "scroll up for vite errors, then re-run this cell."
        )

# 2e. Backend dependencies — same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
# --constraint: `uv pip install` ignores pyproject's `[tool.uv]
# constraint-dependencies` (a project-API setting), so without it torch,
# torchaudio and torchvision resolve on their bare lower bounds and Colab's
# preinstalled torchvision can be left behind a newer torch -- which fails at
# import with "operator torchvision::nms does not exist" (#1357).
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2f. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). PyTorch 2.8+
# ships cuDNN 9; scripts/setup.py places cuDNN 8 libs where the backend
# preloads them from (<repo>/.venv/.../cudnn8_compat). We create the .venv
# directory only so the script has its expected target — no actual venv is
# used on Colab. Non-fatal: without it, transcription falls back to the
# torch-native Whisper backend automatically.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch).
# Imports the backend's own model stack, not just torch — Colab's system
# Python mixes preinstalled and freshly-resolved wheels, and a torchaudio or
# transformers that can't load together only shows up when the model module is
# imported (#1229). Catching it here beats a 5-minute health timeout in cell 5.
run([sys.executable, "-c",
     # Versions FIRST: if the model-stack import below fails, the cell output
     # still shows what was actually installed, which is the single most
     # useful line for diagnosing a Colab environment (#1229).
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, "
     "CUDA available: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")

Repo already present at /content/VoiceStudio — reusing it.

$ apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)



$ /root/.bun/bin/bun --version
frontend/dist already built — skipping.

$ uv pip install --system --no-cache --constraint deploy/torch-constraints.txt .


Using Python 3.13.15 environment at: /usr
Resolved 239 packages in 2.26s
Prepared 79 packages in 33.55s
Uninstalled 17 packages in 8.22s
Installed 79 packages in 373ms
 + addict==2.4.0
 + argostranslate==1.11.0
 + asteroid-filterbanks==0.4.0
 + audioseal==0.2.0
 + av==15.1.0
 + backoff==2.2.1
 - boto3==1.41.5
 + boto3==1.43.109
 - botocore==1.41.5
 + botocore==1.43.109
 + braceexpand==0.1.7
 + csvw==4.1.0
 + ctranslate2==4.8.2
 + curated-tokenizers==0.0.10
 + curated-transformers==0.1.1
 + demucs==4.1.0
 + dlinfo==2.0.0
 + docopt==0.6.2
 + espeakng-loader==0.2.4
 - fastapi==0.141.1
 + fastapi==0.136.3
 + faster-whisper==1.2.1
 + httpx-sse==0.4.3
 + hyperpyyaml==1.2.3
 + isodate==0.7.2
 + julius==0.2.8
 + kittentts==0.8.1 (from https://github.com/KittenML/KittenTTS/releases/download/0.8.1/kittentts-0.8.1-py3-none-any.whl)
 + lameenc==1.8.4
 + language-tags==1.3.1
 + lightning==2.6.6
 - litellm==1.85.7
 + litellm==1.104.0
 + mcp==1.30.0
 + minisbd==0.9.5
 + misaki==0.9.4
 + num2words==0.


$ /usr/bin/python3 /content/VoiceStudio/scripts/setup.py

$ /usr/bin/python3 -c import torch, torchaudio, uvicorn, fastapi, transformers; print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}, CUDA available: {torch.cuda.is_available()}'); from omnivoice.models.omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel; print('Install OK - backend model stack imports cleanly')


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [5]:
import os

# Verification script
out_renderer = "/content/VoiceStudio/frontend/out/renderer"
frontend_dist = "/content/VoiceStudio/frontend/dist"
dist_index = os.path.join(frontend_dist, "index.html")

# Safely create the symlink if it doesn't exist yet
if os.path.exists(out_renderer) and not os.path.exists(frontend_dist):
    print("Creating symlink from out/renderer to dist...")
    os.symlink(out_renderer, frontend_dist)

if os.path.exists(dist_index):
    print(f"SUCCESS: Frontend is correctly mapped and available at {dist_index}!")
else:
    print(f"WARNING: {dist_index} is still missing. Please re-run the build cell.")

SUCCESS: Frontend is correctly mapped and available at /content/VoiceStudio/frontend/dist/index.html!


In [6]:
import os
import shutil

repo_dir = "/content/VoiceStudio"
node_modules_root = os.path.join(repo_dir, "node_modules")
node_modules_frontend = os.path.join(repo_dir, "frontend", "node_modules")

for path in [node_modules_root, node_modules_frontend]:
    if os.path.exists(path):
        print(f"Removing {path}...")
        shutil.rmtree(path)
        print(f"Successfully removed {path}.")
    else:
        print(f"{path} does not exist. (Skipped)")

Removing /content/VoiceStudio/node_modules...
Successfully removed /content/VoiceStudio/node_modules.
/content/VoiceStudio/frontend/node_modules does not exist. (Skipped)


In [7]:
import subprocess
import os

REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Configure environment for Bun
env = os.environ.copy()
bun_dir = os.path.dirname(BUN)
if bun_dir not in env["PATH"]:
    env["PATH"] = bun_dir + os.pathsep + env["PATH"]

# Run the build directly capturing and printing stdout/stderr
print("Running 'bun install --frozen-lockfile' first to restore clean dependencies...")
subprocess.run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, env=env)

print("\nRunning 'bun run --cwd frontend build' and capturing output...")
result = subprocess.run(
    [BUN, "run", "--cwd", "frontend", "build"],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    text=True
)

print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)
print(f"\nExit Code: {result.returncode}")

Running 'bun install --frozen-lockfile' first to restore clean dependencies...

Running 'bun run --cwd frontend build' and capturing output...

--- STDOUT ---
Locale encoding and interpolation: 21 files passed
Locale source keys: 1695 static renderer keys exist in en.json
{
  "missingEntries": 0,
  "byNamespace": {},
  "orphans": {}
}
vite v8.3.0 building ssr environment for production...
transforming...
✓ 56 modules transformed.
rendering chunks...
out/main/repair-mcp-server.js    5.35 kB
out/main/index.js              289.38 kB

✓ built in 98ms
vite v8.3.0 building ssr environment for production...
transforming...
✓ 2 modules transformed.
rendering chunks...
out/preload/index.mjs  6.71 kB

✓ built in 14ms
vite v8.3.0 building client environment for production...
transforming...
✓ 8783 modules transformed.
rendering chunks...
out/renderer/early-error-capture.js                                                        0.94 kB
out/renderer/assets/paoloantinori-DOzNqVNY.jpg                

In [8]:
# ── 3. (Optional) Hugging Face token ────────────────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


No HF_TOKEN Colab Secret found — skipping. (Everything except gated diarization models works without it.)


In [9]:
# ── 4. (Optional, recommended) Pre-download the default voice model ─────────
# The default TTS engine fetches k2-fsa/OmniVoice (a few GB) on its very first
# generation. Downloading it up front makes the first request fast instead of
# a several-minute stall. Safe to re-run: already-downloaded files are reused.
from huggingface_hub import snapshot_download

path = snapshot_download("k2-fsa/OmniVoice")
print("Default TTS model cached at:", path)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Default TTS model cached at: /root/.cache/huggingface/hub/models--k2-fsa--OmniVoice/snapshots/c5fdb5ccb189668d56333f77ba2629f4cd7535f4


In [10]:
# ── 5. Launch the backend ───────────────────────────────────────────────────
# Starts uvicorn on port 3900 and waits for /health. Re-running this cell when
# the backend is already up just reports its status.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

# We force a restart here to ensure the backend loads our newly built frontend assets
print("Force-restarting backend to load updated web assets...")
subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(2)

env = os.environ.copy()
# Headless-server deployment, same as the official Docker image: relaxes
# the desktop-only loopback origin gate so the proxied browser session can
# reach the settings/system routes.
env["OMNIVOICE_SERVER_MODE"] = "1"
# Voices, projects, and generated audio live here. Ephemeral! See the
# troubleshooting cell for persisting it to Google Drive.
env["OMNIVOICE_DATA_DIR"] = "/content/omnivoice_data"
env["PYTHONUNBUFFERED"] = "1"

log = open(LOG_PATH, "ab")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
)
print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

deadline = time.time() + 300
while time.time() < deadline:
    if proc.poll() is not None:
        break  # process died — report below
    info = health()
    if info:
        break
    print(".", end="", flush=True)
    time.sleep(3)
print()

info = health()
if info:
    print(f"Backend is up — {info}")
    if "cuda" not in str(info.get("device", "")):
        print("NOTE: device is not CUDA — generation will be slow. "
              "See cell 1 to enable the GPU runtime.")
else:
    try:
        with open(LOG_PATH, "r", errors="replace") as f:
            tail = "".join(f.readlines()[-40:])
    except OSError:
        tail = "(no log file found)"
    raise SystemExit(
        "Backend did not become healthy within 5 minutes.\n"
        f"--- last lines of {LOG_PATH} ---\n{tail}\n"
        "--- end of log ---\n"
        "Fix the error above (usually a missing dependency: re-run cell 2),\n"
        "then re-run this cell. Still stuck? Attach the log to an issue:\n"
        "  https://github.com/debpalash/VoiceStudio/issues"
    )

Force-restarting backend to load updated web assets...
Backend starting (PID 1516); log: /content/omnivoice_backend.log
....
Backend is up — {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}


In [18]:
# ── 6. Open the app (Cloudflare tunnel + allow its origin) ──────────────────
import os, re, subprocess, time, socket, sys, json, urllib.request

# ⚠️ Match these to your cell 5 (Kaggle usually uses /kaggle/working)
REPO_DIR = "/content/VoiceStudio" if os.path.isdir("/content/VoiceStudio") else "/kaggle/working/VoiceStudio"
DATA_DIR = os.environ.get("OMNIVOICE_DATA_DIR") or (
    "/content/omnivoice_data" if os.path.isdir("/content") else "/kaggle/working/omnivoice_data")
PORT = 3900
LOG_BACKEND = "/tmp/omnivoice_backend.log"
LOG_TUNNEL = "/tmp/cloudflared.log"
BIN = "/tmp/cloudflared"

# 1) download cloudflared
if not os.path.exists(BIN):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", BIN)
    os.chmod(BIN, 0o755)

# 2) start tunnel FIRST so we know its URL (the backend isn't needed for this)
subprocess.run("pkill -f cloudflared", shell=True)
tl = open(LOG_TUNNEL, "w")
subprocess.Popen([BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                  "--http-host-header", f"localhost:{PORT}", "--no-autoupdate"],
                 stdout=tl, stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(LOG_TUNNEL).read())
    if m:
        url = m.group(0); break
if not url:
    raise SystemExit("No tunnel URL.\n" + open(LOG_TUNNEL).read())
print("Tunnel URL:", url)

# 3) restart the backend with the tunnel origin allowed
subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(2)

env = os.environ.copy()
env["OMNIVOICE_SERVER_MODE"] = "1"
env["OMNIVOICE_DATA_DIR"] = DATA_DIR
env["PYTHONUNBUFFERED"] = "1"
env["OMNIVOICE_ALLOWED_ORIGINS"] = url      # <-- the fix (comma-separate if you ever need more)

import secrets
API_KEY = secrets.token_urlsafe(24)
env["OMNIVOICE_API_KEY"] = API_KEY

bl = open(LOG_BACKEND, "ab")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
     "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=REPO_DIR, env=env, stdout=bl, stderr=subprocess.STDOUT)

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

deadline = time.time() + 300
info = None
while time.time() < deadline and proc.poll() is None:
    info = health()
    if info: break
    print(".", end="", flush=True); time.sleep(3)
print()

if info:
    print("✅ Backend up:", info)
    print("\n👉 Open VoiceStudio here:", url)
    print("🔑 Admin API key (paste into the UI's login form when asked):", API_KEY)
    print("Set a share PIN in the app's Settings — anyone with this URL can reach your session.")
else:
    print("❌ Backend didn't start. Last log lines:\n")
    print("".join(open(LOG_BACKEND, errors="replace").readlines()[-40:]))

Tunnel URL: https://slot-chips-placement-fixes.trycloudflare.com
...
✅ Backend up: {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}

👉 Open VoiceStudio here: https://slot-chips-placement-fixes.trycloudflare.com
🔑 Admin API key (paste into the UI's login form when asked): TDHGClkKK4fI8xrfczbhr4d9y1MU1rmI
Set a share PIN in the app's Settings — anyone with this URL can reach your session.


In [12]:
# ── 7. Smoke test: generate speech through the API ──────────────────────────
# Proves the whole stack end-to-end without touching the UI: /health, then a
# real TTS generation played inline. POST /generate returns the WAV directly.
import json
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://127.0.0.1:3900"

with urllib.request.urlopen(f"{BASE}/health", timeout=10) as r:
    print("GET /health ->", json.load(r))

data = urllib.parse.urlencode({
    "text": "VoiceStudio is up and running on Google Colab.",
}).encode()
print("Generating... (first-ever generation loads the model — if you skipped "
      "cell 4 it also downloads it, which can take several minutes)")
try:
    with urllib.request.urlopen(
        urllib.request.Request(f"{BASE}/generate", data=data), timeout=1800
    ) as r:
        wav = r.read()
        print(f"OK — {len(wav)} bytes, duration {r.headers.get('X-Audio-Duration')}s, "
              f"generated in {r.headers.get('X-Gen-Time')}s (seed {r.headers.get('X-Seed')})")
except urllib.error.HTTPError as e:
    detail = e.read().decode("utf-8", errors="replace")
    raise SystemExit(
        f"Generation failed: HTTP {e.code}\n{detail}\n"
        "Check /content/omnivoice_backend.log for the full trace; on a fresh\n"
        "session the usual cause is an interrupted model download — re-run\n"
        "cell 4, then this cell."
    )

OUT = "/content/omnivoice_smoke.wav"
with open(OUT, "wb") as f:
    f.write(wav)

from IPython.display import Audio, display
display(Audio(OUT))


GET /health -> {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}
Generating... (first-ever generation loads the model — if you skipped cell 4 it also downloads it, which can take several minutes)
OK — 117164 bytes, duration 2.44s, generated in 37.11s (seed 1482068657)
